In [3]:
# 1. 确保已安装 web3.py: pip install web3
from web3 import Web3

# --- 配置 ---
# 使用你提供的 Infura RPC URL (它已经包含了你的 Project ID/API Key)
infura_url = "https://mainnet.infura.io/v3/ddf95478b6114a61aa509b9fb6c1bb54"

# 使用之前例子中的交易哈希 (你可以替换成任何你想查询的主网交易哈希)
tx_hash = "0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe"

# --- 连接到以太坊节点 ---
print(f"尝试连接到: {infura_url}")
try:
    w3 = Web3(Web3.HTTPProvider(infura_url))
    if not w3.is_connected():
        print("错误：无法连接到 Infura RPC URL。请检查 URL 是否正确。")
    else:
        print("成功连接到以太坊节点。")

        # --- 获取交易回执 ---
        print(f"正在查询交易哈希: {tx_hash}")
        try:
            receipt = w3.eth.get_transaction_receipt(tx_hash)

            if receipt:
                block_number = receipt['blockNumber']
                block_hash = w3.to_hex(receipt['blockHash']) # 转换为十六进制字符串

                print(f"\n--- 查询结果 ---")
                print(f"交易哈希: {tx_hash}")
                print(f"所属区块号 (Block Number): {block_number}")
                print(f"所属区块哈希 (Block Hash): {block_hash}")

                # 计算清算前需要查询的区块号
                previous_block_number = block_number - 1
                print(f"\n为了获取清算前状态，你应该在 GraphQL 查询中使用区块号: {previous_block_number}")

            else:
                print(f"错误：找不到交易哈希 {tx_hash} 的回执。请确保哈希正确且交易已在主网确认。")

        except Exception as e:
            print(f"查询交易回执时发生错误: {e}")

except Exception as e:
    print(f"连接节点时发生错误: {e}")

尝试连接到: https://mainnet.infura.io/v3/ddf95478b6114a61aa509b9fb6c1bb54
成功连接到以太坊节点。
正在查询交易哈希: 0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe
查询交易回执时发生错误: Transaction with hash: '0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe' not found.


# 使用EtherumScan

In [5]:
import requests
import time # 引入 time 模块以避免触发速率限制

# --- 配置 ---
ETHERSCAN_API_KEY = '85MDKMVQ9BIKM294IQY5N47QV1VN6SBXGP' # 替换成你的 Etherscan API Key
if ETHERSCAN_API_KEY == 'YOUR_ETHERSCAN_API_KEY':
    print("错误：请将 'YOUR_ETHERSCAN_API_KEY' 替换为你的 Etherscan API Key。")
    exit()

tx_hash = "0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe"

# Etherscan API V2 统一基础 URL
BASE_URL = "https://api.etherscan.io/v2/api" # 使用 V2 推荐的统一 URL

# Aave V3 主要部署网络的 Chain ID (根据你提供的列表)
# 你可以根据需要增删
networks_to_check = {
    "Ethereum Mainnet": "1",
    "Polygon Mainnet": "137",
    "Arbitrum One Mainnet": "42161",
    "OP Mainnet": "10",            # Optimism
    "Avalanche C-Chain": "43114",
    "Base Mainnet": "8453",
    "BNB Smart Chain Mainnet": "56" # Aave V3 也部署在 BSC
}

found_network = None
block_number = None
block_hash = None

# --- 遍历不同网络的 Chain ID 尝试查询 ---
for network_name, chain_id in networks_to_check.items():
    print(f"--- 正在尝试查询网络: {network_name} (Chain ID: {chain_id}) ---")
    # 构建 V2 格式的 API URL
    params = {
        'module': 'proxy',
        'action': 'eth_getTransactionReceipt',
        'txhash': tx_hash,
        'apikey': ETHERSCAN_API_KEY,
        'chainid': chain_id # <--- 使用 chainid 参数指定网络
    }

    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status() # 检查 HTTP 错误
        data = response.json()

        # Etherscan API V2 的成功/错误判断可能略有不同，
        # 但通常 result 字段存在且不为 null 表示成功找到回执
        if data.get('result'):
            receipt = data['result']
            # 确保回执不是 null
            if receipt is None:
                 print(f"在 {network_name} 上找到交易哈希，但回执为 null (可能未确认或发生在其他链)。")
                 continue

            block_number_hex = receipt.get('blockNumber')
            current_block_hash = receipt.get('blockHash')

            if block_number_hex and current_block_hash:
                block_number = int(block_number_hex, 16) # 从十六进制转换为整数
                block_hash = current_block_hash
                found_network = network_name
                print(f"成功！在网络 {found_network} 上找到交易。")
                break # 找到后停止搜索
            else:
                 print(f"在 {network_name} 上找到回执，但缺少 blockNumber 或 blockHash。回执内容: {receipt}")

        elif data.get('message') and 'NOTOK' in data.get('message'):
             # 处理 API 返回的明确错误信息
             print(f"在 {network_name} 上查询失败: {data.get('result', data.get('message', '未知 API 错误'))}")
        else:
            # result 为 null 或空 通常意味着该网络没有这个交易
            print(f"在 {network_name} 上未找到交易哈希 {tx_hash} (result 为空或 null)。")

    except requests.exceptions.RequestException as e:
        print(f"请求 {network_name} API 时发生错误: {e}")
    except Exception as e:
        print(f"处理 {network_name} 响应时发生意外错误: {e}")

    # 稍微暂停一下，避免触发速率限制 (Etherscan 免费 Key 每秒 5 次)
    time.sleep(0.3) # 暂停 0.3 秒

# --- 输出最终结果 ---
if found_network and block_number is not None:
    print(f"\n--- 最终结果 ---")
    print(f"交易哈希: {tx_hash}")
    print(f"所属网络: {found_network}")
    print(f"所属区块号 (Block Number): {block_number}")
    print(f"所属区块哈希 (Block Hash): {block_hash}")

    # 计算清算前需要查询的区块号
    previous_block_number = block_number - 1
    print(f"\n为了获取清算前状态，你应该在 GraphQL 查询中使用区块号: {previous_block_number}")
else:
    print(f"\n未能通过 Etherscan API V2 在尝试的网络中找到交易 {tx_hash} 的有效信息。")
    print("请确认交易哈希是否正确，或尝试在 network_endpoints 中添加更多 Aave V3 可能部署的网络及其 Chain ID。")

--- 正在尝试查询网络: Ethereum Mainnet (Chain ID: 1) ---
在 Ethereum Mainnet 上未找到交易哈希 0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe (result 为空或 null)。
--- 正在尝试查询网络: Polygon Mainnet (Chain ID: 137) ---
在 Polygon Mainnet 上未找到交易哈希 0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe (result 为空或 null)。
--- 正在尝试查询网络: Arbitrum One Mainnet (Chain ID: 42161) ---
在 Arbitrum One Mainnet 上未找到交易哈希 0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe (result 为空或 null)。
--- 正在尝试查询网络: OP Mainnet (Chain ID: 10) ---
在 OP Mainnet 上未找到交易哈希 0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe (result 为空或 null)。
--- 正在尝试查询网络: Avalanche C-Chain (Chain ID: 43114) ---
在 Avalanche C-Chain 上未找到交易哈希 0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe (result 为空或 null)。
--- 正在尝试查询网络: Base Mainnet (Chain ID: 8453) ---
成功！在网络 Base Mainnet 上找到交易。

--- 最终结果 ---
交易哈希: 0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe
所属网络: Base Mainn

# 扩展到所有的主网

In [8]:
import requests
import time # 引入 time 模块以避免触发速率限制

# --- 配置 ---
ETHERSCAN_API_KEY = '85MDKMVQ9BIKM294IQY5N47QV1VN6SBXGP' # 替换成你的 Etherscan API Key
if ETHERSCAN_API_KEY == 'YOUR_ETHERSCAN_API_KEY':
    print("错误：请将 'YOUR_ETHERSCAN_API_KEY' 替换为你的 Etherscan API Key。")
    exit()

# 使用你想要查询的交易哈希
tx_hash = "0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe" # 可以替换

# Etherscan API V2 统一基础 URL
BASE_URL = "https://api.etherscan.io/v2/api"

# 包含所有提供的主网及其 Chain ID 的字典
networks_to_check = {
    "Base Mainnet": "8453",
    "Ethereum Mainnet": "1",
    "Abstract Mainnet": "2741",
    "ApeChain Mainnet": "33139",
    "Arbitrum Nova Mainnet": "42170",
    "Arbitrum One Mainnet": "42161",
    "Avalanche C-Chain": "43114",
    "Berachain Mainnet": "80094", # 注意: Berachain 主网可能尚未完全启动或集成API
    "BitTorrent Chain Mainnet": "199",
    "Blast Mainnet": "81457",
    "BNB Smart Chain Mainnet": "56",
    "Celo Mainnet": "42220",
    "Fraxtal Mainnet": "252",
    "Gnosis": "100",
    "HyperEVM Mainnet": "999",
    "Katana Mainnet": "747474",
    "Linea Mainnet": "59144",
    "Mantle Mainnet": "5000",
    "Moonbeam Mainnet": "1284",
    "Moonriver Mainnet": "1285",
    "OP Mainnet": "10",
    "opBNB Mainnet": "204",
    "Polygon Mainnet": "137",
    "Scroll Mainnet": "534352",
    "Sei Mainnet": "1329",
    "Sonic Mainnet": "146",
    "Sophon Mainnet": "50104", # 注意: Etherscan 提示此网络可能弃用
    "Swellchain Mainnet": "1923",
    "Taiko Mainnet": "167000",
    "Unichain Mainnet": "130",
    "World Mainnet": "480",
    "XDC Mainnet": "50",
    "zkSync Mainnet": "324"
    # Testnets 已被排除
}

found_network = None
block_number = None
block_hash = None

# --- 遍历不同网络的 Chain ID 尝试查询 ---
for network_name, chain_id in networks_to_check.items():
    print(f"--- 正在尝试查询网络: {network_name} (Chain ID: {chain_id}) ---")
    # 构建 V2 格式的 API URL 参数
    params = {
        'module': 'proxy',
        'action': 'eth_getTransactionReceipt',
        'txhash': tx_hash,
        'apikey': ETHERSCAN_API_KEY,
        'chainid': chain_id # 使用 chainid 参数指定网络
    }

    try:
        response = requests.get(BASE_URL, params=params)
        response.raise_for_status() # 检查 HTTP 错误
        data = response.json()

        # 检查 API 是否返回了有效结果
        if data.get('result'):
            receipt = data['result']
            if receipt is None:
                 print(f"在 {network_name} 上找到交易哈希，但回执为 null。")
                 continue

            block_number_hex = receipt.get('blockNumber')
            current_block_hash = receipt.get('blockHash')

            if block_number_hex and current_block_hash:
                block_number = int(block_number_hex, 16)
                block_hash = current_block_hash
                found_network = network_name
                print(f"成功！在网络 {found_network} 上找到交易。")
                break # 找到后停止搜索
            else:
                 print(f"在 {network_name} 上找到回执，但缺少 blockNumber 或 blockHash。回执内容: {receipt}")

        elif data.get('message') and ('NOTOK' in data.get('message') or 'No records found' in data.get('message')):
             print(f"在 {network_name} 上查询失败或未找到: {data.get('result', data.get('message', '未知 API 错误'))}")
        else:
            # result 为 null 或空 通常意味着该网络没有这个交易
            print(f"在 {network_name} 上未找到交易哈希 {tx_hash} (result 为空或 null)。")

    except requests.exceptions.RequestException as e:
        print(f"请求 {network_name} API 时发生错误: {e}")
    except Exception as e:
        print(f"处理 {network_name} 响应时发生意外错误: {e}")

    # 稍微暂停一下，避免触发速率限制 (Etherscan 免费 Key 每秒 5 次)
    time.sleep(0.3) # 暂停 0.3 秒

# --- 输出最终结果 ---
if found_network and block_number is not None:
    print(f"\n--- 最终结果 ---")
    print(f"交易哈希: {tx_hash}")
    print(f"所属网络: {found_network}")
    print(f"所属区块号 (Block Number): {block_number}")
    print(f"所属区块哈希 (Block Hash): {block_hash}")

    # 计算清算前需要查询的区块号
    previous_block_number = block_number - 1
    print(f"\n为了获取清算前状态，你应该在 GraphQL 查询中使用区块号: {previous_block_number}")
else:
    print(f"\n未能通过 Etherscan API V2 在尝试的所有主网中找到交易 {tx_hash} 的有效信息。")
    print("请再次确认交易哈希是否正确，或检查是否有主网遗漏。")

--- 正在尝试查询网络: Base Mainnet (Chain ID: 8453) ---
成功！在网络 Base Mainnet 上找到交易。

--- 最终结果 ---
交易哈希: 0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe
所属网络: Base Mainnet
所属区块号 (Block Number): 37495934
所属区块哈希 (Block Hash): 0xcd43a575e120b69613a53a06b6ac245a8354fd4cd2b8aba89b816366144d1ef4

为了获取清算前状态，你应该在 GraphQL 查询中使用区块号: 37495933


# 示例

交易哈希: 0x40e9931a0d22af65521789770565ab3779c697e2cdfe4f7ff45ea101ecabaafe

所属网络: Base Mainnet

所属区块号 (Block Number): 37495934

所属区块哈希 (Block Hash): 0xcd43a575e120b69613a53a06b6ac245a8354fd4cd2b8aba89b816366144d1ef4